# E6: external test on MCT-LTDiag (HCC vs ICC)

**Revision plan, experiment E6 — plan approved 17 Oct 2026 (00 log, section 17, with amendments 17.9 made before
any run).** Answers Reviewer 4 point 2 and Reviewer 2 points 2–4: a real external test with both HCC and ICC, all four
phases, real age and sex, and AUC, sensitivity and specificity instead of HCC sensitivity alone.

**Data:** MCT-LTDiag (Peking Union Medical College Hospital), Harvard Dataverse doi:10.7910/DVN/S3RW15 v3.1, CC0;
HCC 103 + ICC 100, downloaded and checksum-verified by E6a into `liver_project/external_validation/MCT-LTDiag/`.
MCT-LTDiag has no combined HCC-ICC class, so this is not a three-class external test.

**Order of the notebook (nothing is tuned on MCT-LTDiag):**
1. Input checksums; the frozen models must reproduce their saved internal test predictions.
2. Internal orientation reference, computed from the internal portal-venous scans.
3. **Step A — facts check** of every MCT-LTDiag scan (no model, no class labels): grids, intensities, masks, DICOM
   scanner data, and the orientation of each scan. Fixed stop rules.
4. Preprocessing exactly as fixed in the plan: primary (in-plane 0.78 mm, orientation matched to the internal data)
   and the sensitivity variants S1 (no resampling), S2 (1 mm isotropic), S4 (slices reversed), S5 (submitted external
   pipeline's axis handling).
5. Frozen models: phase-dropout CNN (main), original CNN, single-phase CNN, E2 radiomics without shape features,
   CNN + radiomics and full fusion (real age/sex), each fusion with both CNNs.
6. Results: **primary = HCC-vs-ICC AUC of CNN + radiomics with the phase-dropout CNN, primary preprocessing, all
   patients**. Everything else is reported alongside and does not replace it.

**Runtime:** GPU runtime (L4 is enough), about 1–1.5 hours, mostly reading 68 GB of scans and radiomics.
**Safety:** inputs are only read; all outputs go to a new folder `liver_project/revision_2026-10/E6_<time>/`.

In [ ]:
# Cell 1 — Setup: installs, paths, input checks, output folder
import os, sys, io, re, json, time, shutil, tarfile, hashlib, platform, datetime, warnings, subprocess
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    WORK = Path('/content/e6_work')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK', 'monai', 'lightgbm', 'pydicom', 'nibabel'], check=True)
    radiomics_ok = False
    for spec in (['pyradiomics'], ['git+https://github.com/AIM-Harvard/pyradiomics.git']):
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + spec).returncode == 0:
            radiomics_ok = True
            break
    if not radiomics_ok:
        raise RuntimeError('PyRadiomics could not be installed. Send this output for review.')
else:
    ROOT = Path(os.environ['E6_TEST_ROOT'])
    WORK = Path(os.environ['E6_TEST_WORK'])
WORK.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import SimpleITK as sitk
import nibabel as nib
import pydicom
import joblib
from radiomics import featureextractor
from monai.networks.nets import resnet18
from scipy import ndimage
from scipy.stats import beta, norm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
MCT_DIR = PROJECT_DIR / 'external_validation' / 'MCT-LTDiag'
SEED = 42
CLASSES = ['HCC', 'ICC', 'CHCC']
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']
N_IO = 6 if IN_COLAB else 2                       # patients read/preprocessed at the same time
N_JOBS = os.cpu_count() if IN_COLAB else 1        # parallel radiomics processes

INPUTS = {
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_noshape_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_original_predictions': (E3_DIR / 'E3A_original_cnn_predictions.csv', '5c02c3ebda984b6283c1137d85c17c171347b24ae1fac450dfc89c5513d0229b'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'mct_selected': (MCT_DIR / 'metadata' / 'selected_HCC_ICC_patients.csv', 'fdb8d8fa74d8211bbfe45ecb4820415e52a5183ce5d51bcfa4376cfa9ed11e20'),
    'mct_patient_table': (MCT_DIR / 'metadata' / 'meta_info_patient.csv', 'f9f0fc730b1ec75cecb615304aa0f69db30225fb559b460ef78d0fe693220f0d'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
LARGE_INPUTS = {'processed_tar': (PROJECT_DIR / 'processed.tar', 402616320)}
MODEL_FILES = {
    **{f'w3_{f}': PROJECT_DIR / 'checkpoints_w3' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'w4_{f}': PROJECT_DIR / 'checkpoints_w4' / f'fold_{f}_best.pt' for f in range(5)},
    **{f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)},
    **{f'rad_{f}': E2_DIR / 'e2_radiomics_noshape_fold_models' / f'fold_{f}.joblib' for f in range(5)},
}

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

def md5_of(path):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()

problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
for key, (path, size) in LARGE_INPUTS.items():
    if not path.exists() or (IN_COLAB and path.stat().st_size != size):
        problems.append(f'missing or changed size: {path}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import sklearn, scipy, radiomics, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version_VersionString() if hasattr(sitk, 'Version_VersionString') else 'unknown',
            'pyradiomics': getattr(radiomics, '__version__', 'unknown'), 'pydicom': getattr(pydicom, '__version__', 'unknown'),
            'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'device': str(DEVICE), 'cpus': os.cpu_count()}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E6_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=False)
print('Inputs and models checked:', len(INPUTS) + len(LARGE_INPUTS) + len(MODEL_FILES))
print('Versions:', VERSIONS)
print('Output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Frozen models must reproduce their saved internal test predictions; internal orientation reference
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(LARGE_INPUTS['processed_tar'][0]), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E6_TEST_PROCESSED'])

split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
TEST_CT, TEST_LABEL = {}, {}
for pid in test_ids:
    with np.load(PROCESSED_DIR / f'{pid}.npz', allow_pickle=False) as z:
        TEST_CT[pid], TEST_LABEL[pid] = z['ct'].astype(np.uint8), int(z['label'])

def zscore_nonzero_per_phase(x4):          # Week 4, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def make_cnn(in_channels):
    return resnet18(spatial_dims=3, n_input_channels=in_channels, num_classes=3, feed_forward=True)

def load_models(keys, in_channels):
    models = []
    for key in keys:
        checkpoint = torch.load(MODEL_FILES[key], map_location=DEVICE, weights_only=False)
        model = make_cnn(in_channels).to(DEVICE)
        model.load_state_dict(checkpoint['state_dict'])
        model.eval()
        models.append(model)
    return models

CNNS = {'single_phase_CNN': (load_models([f'w3_{f}' for f in range(5)], 1), [2]),
        'original_CNN': (load_models([f'w4_{f}' for f in range(5)], 4), [0, 1, 2, 3]),
        'phase_dropout_CNN': (load_models([f'd_{f}' for f in range(5)], 4), [0, 1, 2, 3])}

@torch.no_grad()
def cnn_probs(models, channels, arrays):
    """Mean softmax over the 5 fold models; arrays: patient -> uint8 (4, 96, 96, 96) in internal axis order."""
    rows = {}
    for pid, ct in arrays.items():
        x = torch.from_numpy(zscore_nonzero_per_phase(ct)[channels]).unsqueeze(0).to(DEVICE)
        rows[pid] = np.mean([torch.softmax(m(x), dim=1).float().cpu().numpy()[0] for m in models], axis=0)
    return pd.DataFrame.from_dict(rows, orient='index', columns=['p_HCC', 'p_ICC', 'p_CHCC'])

def macro_auc3(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

y_test = [TEST_LABEL[p] for p in test_ids]
gate = {}
for name, key, model_tag in [('original_CNN', 'e3_original_predictions', 'original'),
                             ('phase_dropout_CNN', 'e3_d_predictions', 'D_phase_dropout')]:
    saved = pd.read_csv(INPUTS[key][0], dtype={'patient_id': str})
    saved = saved.query("pattern == 'complete' and split == 'test'").set_index('patient_id').loc[test_ids, ['p_HCC', 'p_ICC', 'p_CHCC']]
    ours = cnn_probs(*CNNS[name], {p: TEST_CT[p] for p in test_ids}).loc[test_ids]
    gate[name] = {'max_prob_diff': float(np.abs(ours.to_numpy() - saved.to_numpy()).max()),
                  'macro_auc_ours': macro_auc3(y_test, ours.to_numpy()), 'macro_auc_saved': macro_auc3(y_test, saved.to_numpy())}

internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
rad_models = [joblib.load(MODEL_FILES[f'rad_{f}']) for f in range(5)]
pool_cols = rad_models[0]['feature_pool']
X_int_test = internal.set_index('patient_id').loc[test_ids, pool_cols].to_numpy(dtype=np.float32)
p_int = np.mean([m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X_int_test[:, m['sel_idx']])))
                 for m in rad_models], axis=0)
saved_int = pd.read_csv(INPUTS['e2_noshape_test'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids, ['p_HCC', 'p_ICC', 'p_CHCC']].to_numpy()
gate['radiomics_noshape'] = {'max_prob_diff': float(np.abs(p_int - saved_int).max())}
GATE_OK = (all(gate[n]['max_prob_diff'] < 0.02 and abs(gate[n]['macro_auc_ours'] - gate[n]['macro_auc_saved']) < 0.005
               for n in ['original_CNN', 'phase_dropout_CNN']) and gate['radiomics_noshape']['max_prob_diff'] < 1e-6)
print('Internal reproduction check:', json.dumps(gate, indent=1))
if IN_COLAB and not GATE_OK:
    raise RuntimeError('STOP — the frozen models do not reproduce their saved internal test predictions.')

# ---- internal orientation reference (stored arrays, as Week 2 read them with nibabel) ----
def load_nii(path):                          # Week 2, unchanged
    arr = nib.load(str(path)).get_fdata().astype(np.float32)
    while arr.ndim > 3:
        if arr.shape[-1] == 1:
            arr = arr.squeeze(-1)
        elif arr.shape[-1] == 3:
            arr = arr.mean(axis=-1)
        else:
            arr = arr[..., 0]
    return arr

BONE_0255 = 254                               # >= ~198 HU on the -200..200 -> 0..255 scale

def orientation_descriptor(ct0255_ijk, liver_ijk):
    """Where the liver and the bone are in an (i, j, k) array, as fractions of each axis (no class labels)."""
    li, lj, lk = np.nonzero(liver_ijk)
    if li.size < 500:
        return None
    shape = liver_ijk.shape
    bone = (ct0255_ijk >= BONE_0255) & ~liver_ijk
    bi, bj, _ = np.nonzero(bone)
    area = liver_ijk.sum(axis=(0, 1)).astype(np.float64)
    k0, k1 = int(lk.min()), int(lk.max())
    centre_k = float((np.arange(len(area)) * area).sum() / area.sum())
    return {'liver_i': float(li.mean() / shape[0]), 'liver_j': float(lj.mean() / shape[1]),
            'bone_i': float(bi.mean() / shape[0]) if bi.size >= 200 else np.nan,
            'bone_j': float(bj.mean() / shape[1]) if bj.size >= 200 else np.nan,
            'bulk_k': (centre_k - k0) / max(1, k1 - k0)}

patients_table = pd.read_csv(INPUTS['patient_data'][0], dtype={'patient_id': str})
p_rows = patients_table[patients_table['phase'] == 'P'].drop_duplicates('patient_id')   # non-contrast: bone not confused with contrast

def internal_descriptor(row):
    try:
        return orientation_descriptor(load_nii(DATA_DIR / row['ct_path']), load_nii(DATA_DIR / row['liver_mask_path']) > 0)
    except Exception as error:
        return {'error': repr(error)[:120]}

with ThreadPoolExecutor(8 if IN_COLAB else 2) as pool:
    internal_desc = pd.DataFrame([d or {} for d in pool.map(internal_descriptor, [r for _, r in p_rows.iterrows()])])
internal_desc.insert(0, 'patient_id', p_rows['patient_id'].tolist())
internal_desc.to_csv(OUT_DIR / 'E6_internal_orientation_reference.csv', index=False)
ok = internal_desc.dropna(subset=['liver_i', 'bone_i', 'bone_j'])
REF = {'front_back_axis_is_j': float((abs(ok['bone_j'] - .5) > abs(ok['bone_i'] - .5)).mean()),
       'liver_at_low_i': float((ok['liver_i'] < .5).mean()), 'bone_at_high_j': float((ok['bone_j'] > .5).mean()),
       'liver_bulk_at_low_k': float((ok['bulk_k'] < .5).mean()), 'n': int(len(ok)),
       'median_liver_i': float(ok['liver_i'].median()), 'median_bone_j': float(ok['bone_j'].median()),
       'median_bulk_k': float(ok['bulk_k'].median())}
print('Internal orientation reference:', json.dumps(REF, indent=1))
# Must agree with the 12-patient check in the 00 log (17.9): i = left-right with the liver at low i, spine at high j, bulk at low k
if not (REF['front_back_axis_is_j'] >= .8 and REF['liver_at_low_i'] >= .8 and REF['bone_at_high_j'] >= .8 and REF['liver_bulk_at_low_k'] >= .7):
    raise RuntimeError('STOP — the internal orientation reference differs from the 17 Oct check. Send this output.')

In [ ]:
# Cell 3 — Step A: facts check of every MCT-LTDiag scan (no model, no class labels), orientation decisions, stop rules
SEL = pd.read_csv(INPUTS['mct_selected'][0], dtype=str)
SEL['size'] = SEL['size'].astype(int)
counts = SEL['type'].value_counts().to_dict()
if IN_COLAB and counts != {'HCC': 103, 'ICC': 100}:
    raise RuntimeError(f'STOP — unexpected MCT-LTDiag selection {counts}.')
WANTED = {'NIFTI/nc.nii.gz': 'P', 'NIFTI/art.nii.gz': 'C1', 'NIFTI/pvp.nii.gz': 'C2', 'NIFTI/delay.nii.gz': 'C3',
          'mask_pvp.nii.gz': 'tumor', 'liver_mask_pvp.nii.gz': 'liver'}
DICOM_FIELDS = ['SeriesDescription', 'Manufacturer', 'ManufacturerModelName', 'SliceThickness', 'PixelSpacing', 'KVP',
                'ConvolutionKernel', 'SeriesInstanceUID']
TAR_TMP, NII_DIR = WORK / 'tar', WORK / 'nifti'
TAR_TMP.mkdir(exist_ok=True)
NII_DIR.mkdir(exist_ok=True)

def window_0255(hu):                          # PLC-CECT release convention: -200..200 HU -> 0..255
    return ((np.clip(hu, -200.0, 200.0) + 200.0) / 400.0 * 255.0).astype(np.float32)

def ijk(img):                                 # SimpleITK array (k, j, i) -> stored (i, j, k) order, as nibabel / Week 2
    arr = sitk.GetArrayFromImage(img)
    return np.ascontiguousarray(np.transpose(arr, (2, 1, 0)))

def grid(img):
    return img.GetSize(), img.GetSpacing(), img.GetOrigin(), img.GetDirection()

def same_grid(a, b):
    return (tuple(a[0]) == tuple(b[0]) and np.allclose(a[1], b[1], rtol=1e-3) and np.allclose(a[2], b[2], atol=0.5)
            and np.allclose(a[3], b[3], atol=1e-3))

def onto(img, reference, is_mask):
    return sitk.Resample(img, reference, sitk.Transform(), sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear,
                         0 if is_mask else -1024)

def nifti_facts(folder):
    imgs = {k: sitk.ReadImage(str(folder / f'{k}.nii.gz')) for k in PHASE_ORDER + ['tumor', 'liver']}
    ref = grid(imgs['C2'])
    f = {'size': list(ref[0]), 'spacing': [round(float(s), 4) for s in ref[1]], 'direction': [round(float(d), 3) for d in ref[3]],
         'off_grid': [k for k in imgs if k != 'C2' and not same_grid(grid(imgs[k]), ref)]}
    mm = sitk.MinimumMaximumImageFilter()
    for k in PHASE_ORDER:
        mm.Execute(imgs[k])
        sample = sitk.GetArrayViewFromImage(imgs[k])[::2, ::4, ::4]
        f[f'{k}_min'], f[f'{k}_max'] = float(mm.GetMinimum()), float(mm.GetMaximum())
        f[f'{k}_p50'] = float(np.percentile(sample, 50))
    mins, maxs = [f[f'{k}_min'] for k in PHASE_ORDER], [f[f'{k}_max'] for k in PHASE_ORDER]
    f['intensity_kind'] = ('HU' if max(mins) <= -500 else '0-255' if min(mins) >= 0 and max(maxs) <= 255 else 'unexpected')
    tumor, liver = sitk.GetArrayFromImage(imgs['tumor']), sitk.GetArrayFromImage(imgs['liver'])
    f['tumor_labels'] = [float(v) for v in np.unique(tumor)[:10]]
    f['liver_labels'] = [float(v) for v in np.unique(liver)[:10]]
    f['tumor_voxels'], f['liver_voxels'] = int((tumor > 0).sum()), int((liver > 0).sum())
    f['tumor_components'] = int(ndimage.label(tumor > 0)[1])
    f['tumor_volume_ml'] = f['tumor_voxels'] * float(np.prod(ref[1])) / 1000.0
    f['tumor_outside_liver_fraction'] = float(((tumor > 0) & (liver == 0)).sum() / max(1, (tumor > 0).sum()))
    plain = imgs['P'] if 'P' not in f['off_grid'] else onto(imgs['P'], imgs['C2'], is_mask=False)
    liver_img = imgs['liver'] if 'liver' not in f['off_grid'] else onto(imgs['liver'], imgs['C2'], is_mask=True)
    ct = ijk(plain)
    descriptor = orientation_descriptor(window_0255(ct) if f['intensity_kind'] == 'HU' else ct, ijk(liver_img) > 0)
    if descriptor is None or not np.isfinite(descriptor['bone_i']):
        raise RuntimeError('liver mask or bone too small to determine orientation')
    f.update(descriptor)
    f['nib_equals_sitk_ijk'] = bool(np.allclose(load_nii(folder / 'P.nii.gz'), ijk(imgs['P']).astype(np.float32), atol=1e-3))
    return f

def stage(row):
    pid = row['patient_id']
    folder = NII_DIR / pid
    done = folder / 'facts.json'
    if done.exists():
        return json.loads(done.read_text())
    folder.mkdir(exist_ok=True)
    facts, series = {'patient_id': pid, 'technical_failure': ''}, {}
    local = TAR_TMP / row['name']
    try:
        shutil.copyfile(MCT_DIR / f"tar_{row['type']}" / row['name'], local)
        if md5_of(local) != row['md5']:
            raise RuntimeError('checksum mismatch after copying from Drive')
        found = set()
        with tarfile.open(local, 'r:') as tf:
            for m in tf:
                if not m.isfile():
                    continue
                name = m.name[2:] if m.name.startswith('./') else m.name
                if name in WANTED:
                    (folder / f'{WANTED[name]}.nii.gz').write_bytes(tf.extractfile(m).read())   # fixed local names only
                    found.add(name)
                elif name.startswith('DICOM/'):
                    key = name.rsplit('/', 1)[0]
                    if key not in series:
                        try:
                            ds = pydicom.dcmread(io.BytesIO(tf.extractfile(m).read()), stop_before_pixels=True, force=True)
                            series[key] = {field: str(getattr(ds, field, '')) for field in DICOM_FIELDS}
                        except Exception as error:
                            series[key] = {'error': repr(error)[:100]}
                        series[key]['files'] = 0
                    series[key]['files'] += 1
        if set(WANTED) - found:
            raise RuntimeError(f'missing in tar: {sorted(set(WANTED) - found)}')
        facts.update(nifti_facts(folder))
        if facts['tumor_voxels'] == 0:
            facts['technical_failure'] = 'empty tumour mask'
    except Exception as error:
        facts['technical_failure'] = repr(error)[:200]
    finally:
        local.unlink(missing_ok=True)
    facts['dicom_series'] = list(series.values())
    done.write_text(json.dumps(facts, default=float))
    return facts

t0 = time.time()
rows = [r for _, r in SEL.iterrows()]
FACTS = []
with ThreadPoolExecutor(N_IO) as pool:
    for n, f in enumerate(pool.map(stage, rows), start=1):
        FACTS.append(f)
        if n % 20 == 0 or n == len(rows):
            print(f'  facts check {n}/{len(rows)} ({(time.time() - t0) / 60:.1f} min)', flush=True)

dicom_rows = [{'patient_id': f['patient_id'], **s} for f in FACTS for s in f.get('dicom_series', [])]
pd.DataFrame(dicom_rows).to_csv(OUT_DIR / 'E6_dicom_series.csv', index=False)
facts_df = pd.DataFrame([{k: v for k, v in f.items() if k != 'dicom_series'} for f in FACTS])
TECH_FAIL = facts_df.loc[facts_df['technical_failure'] != '', ['patient_id', 'technical_failure']]
INCLUDED = facts_df.loc[facts_df['technical_failure'] == '', 'patient_id'].tolist()
kinds = Counter(facts_df.loc[facts_df['technical_failure'] == '', 'intensity_kind'])
print(f'Included {len(INCLUDED)}/{len(SEL)}; technical failures: {TECH_FAIL.to_dict("records")}')
print('Intensity kind:', dict(kinds), '| phases off the portal-venous grid:',
      int(facts_df['off_grid'].map(lambda v: len(v) if isinstance(v, list) else 0).gt(0).sum()), 'patients')
if kinds.get('unexpected', 0):
    raise RuntimeError('STOP — some scans are neither HU nor 0-255 (rule 17.3-1). Send this output.')
if IN_COLAB and not facts_df.loc[facts_df['technical_failure'] == '', 'nib_equals_sitk_ijk'].all():
    raise RuntimeError('STOP — SimpleITK (i, j, k) arrays differ from nibabel arrays.')

# ---- orientation decisions (amendment 17.9-1): match each scan to the internal anatomy ----
def decide(d):
    # front-back axis = the in-plane axis on which the bone (spine) is off-centre; internally that is j
    off_i, off_j = abs(d['bone_i'] - .5), abs(d['bone_j'] - .5)
    out = {'transpose': off_i > off_j, 'conf_transpose': max(off_i, off_j) >= .05 and abs(off_i - off_j) >= .03}
    li, bj = (d['liver_j'], d['bone_i']) if out['transpose'] else (d['liver_i'], d['bone_j'])
    out['flip_i'], out['conf_flip_i'] = bool(li > .5), abs(li - .5) >= .05
    out['flip_j'] = bool(bj < .5) if np.isfinite(bj) else False
    out['conf_flip_j'] = bool(np.isfinite(bj) and abs(bj - .5) >= .05)
    out['flip_k'], out['conf_flip_k'] = bool(d['bulk_k'] > .5), abs(d['bulk_k'] - .5) >= .03
    return out

ORIENT = {f['patient_id']: decide(f) for f in FACTS if f['patient_id'] in INCLUDED}
orient_df = pd.DataFrame.from_dict(ORIENT, orient='index')
orientation_summary = {}
for part in ['transpose', 'flip_i', 'flip_j', 'flip_k']:
    confident = orient_df[orient_df[f'conf_{part}']]
    majority = bool(confident[part].mean() > .5) if len(confident) else False
    share = float((confident[part] == majority).mean()) if len(confident) else 0.0
    unclear = orient_df.index[~orient_df[f'conf_{part}']]
    orient_df.loc[unclear, part] = majority
    orientation_summary[part] = {'applied_to': int(orient_df[part].sum()), 'unclear_set_to_majority': int(len(unclear)),
                                 'majority': majority, 'agreement_among_clear': round(share, 3)}
    if len(unclear) > 0.10 * len(orient_df):                 # scans may differ from each other; too many unclear ones may not
        raise RuntimeError(f'STOP — orientation part {part!r} is unclear for {len(unclear)} scans (>10%). Send this output.')
ORIENT = orient_df[['transpose', 'flip_i', 'flip_j', 'flip_k']].astype(bool).to_dict('index')
orient_df.to_csv(OUT_DIR / 'E6_orientation_decisions.csv')
facts_df.to_csv(OUT_DIR / 'E6_facts_per_patient.csv', index=False)

ok = facts_df[facts_df['technical_failure'] == '']
dicom = pd.DataFrame(dicom_rows)
FACTS_SUMMARY = {
    'in_plane_spacing_mm': [float(ok['spacing'].map(lambda s: s[0]).min()), float(ok['spacing'].map(lambda s: s[0]).median()),
                            float(ok['spacing'].map(lambda s: s[0]).max())],
    'slice_spacing_mm': [float(ok['spacing'].map(lambda s: s[2]).min()), float(ok['spacing'].map(lambda s: s[2]).median()),
                         float(ok['spacing'].map(lambda s: s[2]).max())],
    'tumour_components': ok['tumor_components'].value_counts().sort_index().to_dict(),
    'tumour_volume_ml_median': float(ok['tumor_volume_ml'].median()),
    'manufacturers': dicom['Manufacturer'].value_counts().to_dict() if 'Manufacturer' in dicom else {},
    'dicom_slice_thickness': dicom['SliceThickness'].value_counts().head(6).to_dict() if 'SliceThickness' in dicom else {},
    'kvp': dicom['KVP'].value_counts().head(6).to_dict() if 'KVP' in dicom else {},
}
print('Facts:', json.dumps(FACTS_SUMMARY, indent=1, default=str))
print('Orientation decisions (amendment 17.9-1):', json.dumps(orientation_summary, indent=1))

In [ ]:
# Cell 4 — Preprocessing: primary and sensitivity variants (internal Week 2 crop rules)
INPLANE_MM = 400.0 / 512                       # internal physical pixel size (512 matrix, 400 mm field of view)
CROP_SIZE = 96
VARIANTS = ['primary', 'S1_native', 'S2_iso1mm', 'S4_reversed_slices', 'S5_week10g_axes']
CROP_DIR = WORK / 'crops'
CROP_DIR.mkdir(exist_ok=True)

def compute_center_of_mass(mask):             # Week 2, unchanged
    coords = np.array(np.where(mask > 0))
    if coords.shape[1] == 0:
        return None
    return tuple(int(round(c)) for c in coords.mean(axis=1))

def crop_cube(volume, center, size):          # Week 2, unchanged
    half = size // 2
    shape = volume.shape
    starts = [c - half for c in center]
    ends = [c + size - half for c in center]
    pad_before = [max(0, -s) for s in starts]
    pad_after = [max(0, e - shape[i]) for i, e in enumerate(ends)]
    s0, s1, s2 = (max(0, s) for s in starts)
    e0, e1, e2 = (min(shape[i], e) for i, e in enumerate(ends))
    cropped = volume[s0:e0, s1:e1, s2:e2]
    if any(pad_before) or any(pad_after):
        cropped = np.pad(cropped, [(pad_before[i], pad_after[i]) for i in range(3)], mode='constant', constant_values=0)
    return cropped

def resample(img, spacing, is_mask):
    size = [max(1, int(round(sz * sp / ts))) for sz, sp, ts in zip(img.GetSize(), img.GetSpacing(), spacing)]
    f = sitk.ResampleImageFilter()
    f.SetOutputSpacing(tuple(float(s) for s in spacing))
    f.SetSize(size)
    f.SetOutputOrigin(img.GetOrigin())
    f.SetOutputDirection(img.GetDirection())
    f.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    f.SetDefaultPixelValue(0 if is_mask else -1024)
    return f.Execute(img)

def apply_orientation(a, o):
    if o['transpose']:
        a = np.transpose(a, (1, 0, 2))
    if o['flip_i']:
        a = a[::-1]
    if o['flip_j']:
        a = a[:, ::-1]
    if o['flip_k']:
        a = a[:, :, ::-1]
    return np.ascontiguousarray(a)

def crops_from(arrays):
    liver, tumor = arrays['liver'] > 0, arrays['tumor'] > 0
    center = compute_center_of_mass(tumor) or compute_center_of_mass(liver)
    ct4 = np.stack([np.clip(crop_cube(arrays[p] * liver, center, CROP_SIZE), 0, 255).astype(np.uint8) for p in PHASE_ORDER])
    tumor_crop = (crop_cube(tumor.astype(np.uint8), center, CROP_SIZE) > 0).astype(np.uint8)
    meta = {'shape': list(tumor.shape), 'center': list(center), 'tumor_in_crop': float(tumor_crop.sum() / max(1, tumor.sum()))}
    return ct4, tumor_crop, meta

def preprocess(pid):
    target = CROP_DIR / f'{pid}.npz'
    meta_file = CROP_DIR / f'{pid}.json'
    if target.exists() and meta_file.exists():
        return json.loads(meta_file.read_text())
    folder = NII_DIR / pid
    facts = next(f for f in FACTS if f['patient_id'] == pid)
    imgs = {k: sitk.ReadImage(str(folder / f'{k}.nii.gz')) for k in PHASE_ORDER + ['tumor', 'liver']}
    for k in facts['off_grid']:                                  # rule 17.3-2
        imgs[k] = onto(imgs[k], imgs['C2'], is_mask=k in ('tumor', 'liver'))
    hu = facts['intensity_kind'] == 'HU'
    sz = imgs['C2'].GetSpacing()[2]
    settings = {'primary': (INPLANE_MM, INPLANE_MM, sz), 'S1_native': None, 'S2_iso1mm': (1.0, 1.0, 1.0),
                'S5_week10g_axes': (1.0, 1.0, 1.0)}
    out, meta = {}, {}
    for variant, spacing in settings.items():
        vols = imgs if spacing is None else {k: resample(v, spacing, k in ('tumor', 'liver')) for k, v in imgs.items()}
        if variant == 'S5_week10g_axes':
            arrays = {k: sitk.GetArrayFromImage(v).astype(np.float32) for k, v in vols.items()}   # (k, j, i), as Week 10g
        else:
            arrays = {k: apply_orientation(ijk(v).astype(np.float32), ORIENT[pid]) for k, v in vols.items()}
        for p in PHASE_ORDER:
            arrays[p] = window_0255(arrays[p]) if hu else arrays[p]
        out[f'{variant}_ct'], out[f'{variant}_tumor'], meta[variant] = crops_from(arrays)
        if variant == 'primary':
            flipped = {k: np.ascontiguousarray(v[:, :, ::-1]) for k, v in arrays.items()}
            out['S4_reversed_slices_ct'], out['S4_reversed_slices_tumor'], meta['S4_reversed_slices'] = crops_from(flipped)
    np.savez_compressed(target, **out)
    meta_file.write_text(json.dumps(meta))
    return meta

t0 = time.time()
CROP_META = {}
with ThreadPoolExecutor(N_IO) as pool:
    for n, (pid, meta) in enumerate(zip(INCLUDED, pool.map(preprocess, INCLUDED)), start=1):
        CROP_META[pid] = meta
        if n % 20 == 0 or n == len(INCLUDED):
            print(f'  preprocessed {n}/{len(INCLUDED)} ({(time.time() - t0) / 60:.1f} min)', flush=True)

crop_rows = [{'patient_id': pid, 'variant': v, **m} for pid, metas in CROP_META.items() for v, m in metas.items()]
pd.DataFrame(crop_rows).to_csv(OUT_DIR / 'E6_crop_geometry.csv', index=False)
(OUT_DIR / 'crops_primary').mkdir()
for pid in INCLUDED:
    with np.load(CROP_DIR / f'{pid}.npz') as z:
        np.savez_compressed(OUT_DIR / 'crops_primary' / f'{pid}.npz', ct=z['primary_ct'], tumor_mask=z['primary_tumor'])
geo = pd.DataFrame(crop_rows)
print('Tumour fraction inside the 96^3 crop (median by variant):', geo.groupby('variant')['tumor_in_crop'].median().round(3).to_dict())

In [ ]:
# Cell 5 — Frozen CNNs on every variant
def load_variant(variant):
    arrays = {}
    for pid in INCLUDED:
        with np.load(CROP_DIR / f'{pid}.npz') as z:
            arrays[pid] = z[f'{variant}_ct']
    return arrays

PROBS = {}                                    # (model, variant) -> DataFrame indexed by patient
t0 = time.time()
for variant in VARIANTS:
    arrays = load_variant(variant)
    for name, (models, channels) in CNNS.items():
        PROBS[(name, variant)] = cnn_probs(models, channels, arrays).loc[INCLUDED]
    print(f'  CNNs done for {variant} ({(time.time() - t0) / 60:.1f} min)', flush=True)

In [ ]:
# Cell 6 — Radiomics without shape features (E2 models) on every variant (Week 5 extraction: 0-255 crops, unit spacing)
def make_extractor():
    extractor = featureextractor.RadiomicsFeatureExtractor(binWidth=5)
    extractor.disableAllImageTypes()
    extractor.enableImageTypeByName('Original')
    extractor.disableAllFeatures()
    for feature_class in ('shape', 'firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm'):
        extractor.enableFeatureClassByName(feature_class)
    return extractor

def extract_patient(pid, variant, crop_dir):
    extractor = make_extractor()
    with np.load(crop_dir / f'{pid}.npz') as z:
        ct4, tumor = z[f'{variant}_ct'], z[f'{variant}_tumor']
    features = {}
    if tumor.sum() == 0:
        return pid, features
    mask_img = sitk.GetImageFromArray(tumor.astype(np.uint8))
    for i, phase in enumerate(PHASE_ORDER):
        try:
            for key, value in extractor.execute(sitk.GetImageFromArray(ct4[i].astype(np.float32)), mask_img).items():
                if not key.startswith('diagnostics'):
                    try:
                        features[f'{phase}_{key}'] = float(value)
                    except (TypeError, ValueError):
                        pass
        except Exception as error:
            print(f'  radiomics failed for {pid}/{phase}: {error}')
    return pid, features

RAD = {}
t0 = time.time()
for variant in VARIANTS:
    jobs = [(pid, variant, CROP_DIR) for pid in INCLUDED]
    results = None
    if N_JOBS > 1:
        try:
            results = joblib.Parallel(n_jobs=N_JOBS)(joblib.delayed(extract_patient)(*job) for job in jobs)
        except Exception as error:                       # same results, only slower
            print(f'  parallel radiomics failed ({error!r:.120}); continuing one patient at a time')
    if results is None:
        results = [extract_patient(*job) for job in jobs]
    RAD[variant] = pd.DataFrame.from_dict(dict(results), orient='index').reindex(INCLUDED)
    RAD[variant].to_csv(OUT_DIR / f'E6_radiomics_features_{variant}.csv')
    print(f'  radiomics done for {variant}: {RAD[variant].shape} ({(time.time() - t0) / 60:.1f} min)', flush=True)

def radiomics_probs(features):
    per_fold, imputed, total = [], 0, 0
    for f, m in enumerate(rad_models):
        names = m['sel_names']
        train_ids = [p for p in dev_ids if p not in set(fold_splits[f]['val_ids'])]
        medians = internal.set_index('patient_id').loc[train_ids, names].median(axis=0, skipna=True)
        X = features.reindex(columns=names).to_numpy(dtype=np.float64)
        missing = ~np.isfinite(X)
        imputed += int(missing.sum())
        total += X.size
        X[missing] = np.broadcast_to(medians.to_numpy(), X.shape)[missing]
        per_fold.append(m['clf'].predict_proba(m['scaler'].transform(m['imputer'].transform(X.astype(np.float32)))))
    return pd.DataFrame(np.mean(per_fold, axis=0), index=features.index, columns=['p_HCC', 'p_ICC', 'p_CHCC']), imputed / total

IMPUTED = {}
for variant in VARIANTS:
    PROBS[('radiomics_noshape', variant)], IMPUTED[variant] = radiomics_probs(RAD[variant])
print('Radiomics cells filled with training medians (should be ~0):', {k: round(v, 4) for k, v in IMPUTED.items()})

In [ ]:
# Cell 7 — Fusion exactly as fitted internally (E2/E3/E5), with the real MCT-LTDiag age and sex
W4 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC']
W5 = ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
FUSIONS = {'cnn_plus_radiomics': W4 + W5, 'full_fusion': W4 + W5 + ['age', 'gender']}

oof_original = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str})
d_pred = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
d_oof = d_pred.query("pattern == 'complete' and split == 'oof'").set_index('patient_id')[['p_HCC', 'p_ICC', 'p_CHCC']]
oof_dropout = oof_original.copy()
oof_dropout[W4] = d_oof.loc[oof_dropout['patient_id']].to_numpy()
OOF_BY_CNN = {'original_CNN': oof_original, 'phase_dropout_CNN': oof_dropout}

def prepare(X):
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X

def fit_meta_folds(columns, oof):
    fitted = []
    for f in range(5):
        train = oof['fold'].astype(int).ne(f)
        X = prepare(oof.loc[train, columns].to_numpy())
        imputer = SimpleImputer(strategy='median').fit(X)
        scaler = StandardScaler().fit(imputer.transform(X))
        model = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED)
        fitted.append((imputer, scaler, model.fit(scaler.transform(imputer.transform(X)), oof.loc[train, 'label'].astype(int))))
    return fitted

def parse_age(value):
    match = re.fullmatch(r'0*(\d+)\s*[Yy]?', str(value).strip())
    return float(match.group(1)) if match else np.nan

clin = SEL.set_index('patient_id').loc[INCLUDED]
AGE = clin['patient_age'].map(parse_age).to_numpy()
SEX = clin['patient_sex'].str.strip().str.upper().map({'M': 1.0, 'F': 0.0}).to_numpy()   # internal coding
print(f'Clinical: age observed {np.isfinite(AGE).sum()}/{len(INCLUDED)} (median {np.nanmedian(AGE):.0f}), '
      f'sex observed {np.isfinite(SEX).sum()} (male {int((SEX == 1).sum())}, female {int((SEX == 0).sum())})')

FITTED = {(cnn, fusion): fit_meta_folds(cols, oof) for cnn, oof in OOF_BY_CNN.items() for fusion, cols in FUSIONS.items()}
for variant in VARIANTS:
    for (cnn_name, fusion_name), fitted in FITTED.items():
        frame = pd.DataFrame(index=INCLUDED)
        frame[W4] = PROBS[(cnn_name, variant)].loc[INCLUDED].to_numpy()
        frame[W5] = PROBS[('radiomics_noshape', variant)].loc[INCLUDED].to_numpy()
        frame['age'], frame['gender'] = AGE, SEX
        X = prepare(frame[FUSIONS[fusion_name]].to_numpy())
        probs = np.mean([model.predict_proba(scaler.transform(imputer.transform(X))) for imputer, scaler, model in fitted], axis=0)
        PROBS[(f'{fusion_name} [{cnn_name}]', variant)] = pd.DataFrame(probs, index=INCLUDED, columns=['p_HCC', 'p_ICC', 'p_CHCC'])
print('Models x variants:', len(PROBS))

In [ ]:
# Cell 8 — Results: AUC (DeLong and bootstrap CIs), sensitivity, specificity, confusion tables; saved outputs
PRIMARY = ('cnn_plus_radiomics [phase_dropout_CNN]', 'primary')
MODEL_ORDER = ['cnn_plus_radiomics [phase_dropout_CNN]', 'full_fusion [phase_dropout_CNN]', 'phase_dropout_CNN',
               'radiomics_noshape', 'original_CNN', 'cnn_plus_radiomics [original_CNN]', 'full_fusion [original_CNN]',
               'single_phase_CNN']
N_BOOT = 2000
TRUE = SEL.set_index('patient_id').loc[INCLUDED, 'type']
Y = (TRUE == 'HCC').astype(int).to_numpy()                   # HCC = positive class
PRIOR = SEL.set_index('patient_id').loc[INCLUDED, 'History of chemotherapy (Y/N)'].fillna('').str.strip().str.upper().str.startswith('Y').to_numpy()
SUBSETS = {'all': np.ones(len(INCLUDED), bool), 'no_prior_treatment': ~PRIOR}

def exact_ci(k, n):
    low = 0.0 if k == 0 else float(beta.ppf(0.025, k, n - k + 1))
    high = 1.0 if k == n else float(beta.ppf(0.975, k + 1, n - k))
    return low, high

def compute_midrank(values):
    order = np.argsort(values)
    sorted_values = values[order]
    n_values = len(values)
    midranks = np.zeros(n_values, dtype=np.float64)
    index = 0
    while index < n_values:
        end = index
        while end < n_values and sorted_values[end] == sorted_values[index]:
            end += 1
        midranks[index:end] = 0.5 * (index + end - 1) + 1
        index = end
    output = np.empty(n_values, dtype=np.float64)
    output[order] = midranks
    return output

def delong_components(y, scores):
    pos, neg = scores[y == 1], scores[y == 0]
    m, n = len(pos), len(neg)
    tx, ty, tz = compute_midrank(pos), compute_midrank(neg), compute_midrank(np.concatenate([pos, neg]))
    auc = tz[:m].sum() / m / n - (m + 1.0) / 2.0 / n
    return auc, (tz[:m] - tx) / n, 1.0 - (tz[m:] - ty) / m

def delong_ci(y, s):
    auc, v01, v10 = delong_components(y, s)
    se = np.sqrt(np.var(v01, ddof=1) / len(v01) + np.var(v10, ddof=1) / len(v10))
    return float(auc), float(max(0.0, auc - 1.96 * se)), float(min(1.0, auc + 1.96 * se))

def delong_paired_p(y, s1, s2):
    a1, v01a, v10a = delong_components(y, s1)
    a2, v01b, v10b = delong_components(y, s2)
    cov = np.cov(np.vstack([v01a, v01b])) / len(v01a) + np.cov(np.vstack([v10a, v10b])) / len(v10a)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    return 1.0 if var <= 0 else float(2 * norm.sf(abs(a1 - a2) / np.sqrt(var)))

rng = np.random.default_rng(SEED)
BOOT = [rng.integers(0, len(INCLUDED), len(INCLUDED)) for _ in range(N_BOOT)]

def score(probs):
    p = probs.loc[INCLUDED].to_numpy()
    return p[:, 0] / np.clip(p[:, 0] + p[:, 1], 1e-12, None), p.argmax(axis=1)

def boot_auc(y, s):
    vals = [roc_auc_score(y[b], s[b]) for b in BOOT if 0 < y[b].sum() < len(b)]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

rows = []
for (model, variant), probs in PROBS.items():
    s_all, pred_all = score(probs)
    for subset, keep in SUBSETS.items():
        if subset != 'all' and variant != 'primary':
            continue
        y, s, pred = Y[keep], s_all[keep], pred_all[keep]
        auc, d_low, d_high = delong_ci(y, s)
        b_low, b_high = boot_auc(y, s) if subset == 'all' else (np.nan, np.nan)
        k_sens, n_hcc = int(((pred == 0) & (y == 1)).sum()), int((y == 1).sum())
        k_spec, n_icc = int(((pred == 1) & (y == 0)).sum()), int((y == 0).sum())
        rows.append({'model': model, 'variant': variant, 'subset': subset, 'n': int(len(y)), 'n_HCC': n_hcc, 'n_ICC': n_icc,
                     'auc': auc, 'auc_delong_low': d_low, 'auc_delong_high': d_high, 'auc_boot_low': b_low, 'auc_boot_high': b_high,
                     'hcc_sensitivity': k_sens / n_hcc, 'sens_ci_low': exact_ci(k_sens, n_hcc)[0], 'sens_ci_high': exact_ci(k_sens, n_hcc)[1],
                     'icc_specificity': k_spec / n_icc, 'spec_ci_low': exact_ci(k_spec, n_icc)[0], 'spec_ci_high': exact_ci(k_spec, n_icc)[1],
                     **{f'HCC_pred_{c}': int(((pred == i) & (y == 1)).sum()) for i, c in enumerate(CLASSES)},
                     **{f'ICC_pred_{c}': int(((pred == i) & (y == 0)).sum()) for i, c in enumerate(CLASSES)}})
RESULTS = pd.DataFrame(rows)
RESULTS.to_csv(OUT_DIR / 'E6_results.csv', index=False)

PAIRS = [('phase_dropout_CNN', 'original_CNN'), ('full_fusion [phase_dropout_CNN]', 'cnn_plus_radiomics [phase_dropout_CNN]'),
         ('radiomics_noshape', 'phase_dropout_CNN'), ('cnn_plus_radiomics [phase_dropout_CNN]', 'radiomics_noshape')]
pair_rows = []
for a, b in PAIRS:
    sa, _ = score(PROBS[(a, 'primary')]); sb, _ = score(PROBS[(b, 'primary')])
    diffs = [roc_auc_score(Y[i], sa[i]) - roc_auc_score(Y[i], sb[i]) for i in BOOT if 0 < Y[i].sum() < len(i)]
    pair_rows.append({'model_a': a, 'model_b': b, 'auc_a': roc_auc_score(Y, sa), 'auc_b': roc_auc_score(Y, sb),
                      'difference': roc_auc_score(Y, sa) - roc_auc_score(Y, sb), 'boot_low': float(np.percentile(diffs, 2.5)),
                      'boot_high': float(np.percentile(diffs, 97.5)), 'delong_p': delong_paired_p(Y, sa, sb)})
PAIRED = pd.DataFrame(pair_rows)
PAIRED.to_csv(OUT_DIR / 'E6_paired_differences.csv', index=False)

long = []
for (model, variant), probs in PROBS.items():
    t = probs.loc[INCLUDED].copy()
    t.insert(0, 'patient_id', t.index)
    t.insert(0, 'variant', variant)
    t.insert(0, 'model', model)
    t['true_class'] = TRUE.to_numpy()
    t['prior_treatment'] = PRIOR
    long.append(t)
pd.concat(long, ignore_index=True).to_csv(OUT_DIR / 'E6_predictions.csv', index=False)

prim = RESULTS[(RESULTS['model'] == PRIMARY[0]) & (RESULTS['variant'] == PRIMARY[1]) & (RESULTS['subset'] == 'all')].iloc[0]
summary = {'run_id': RUN_ID, 'notebook': 'E6_external_MCT-LTDiag_HCC_vs_ICC.ipynb', 'versions': VERSIONS,
           'input_sha256': input_hashes, 'internal_reproduction_check': gate, 'internal_orientation_reference': REF,
           'included': len(INCLUDED), 'technical_failures': TECH_FAIL.to_dict('records'), 'facts': FACTS_SUMMARY,
           'orientation': orientation_summary, 'radiomics_imputed_fraction': IMPUTED,
           'primary': {'model': PRIMARY[0], 'variant': PRIMARY[1], 'auc': prim['auc'],
                       'delong_ci': [prim['auc_delong_low'], prim['auc_delong_high']],
                       'bootstrap_ci': [prim['auc_boot_low'], prim['auc_boot_high']],
                       'hcc_sensitivity': prim['hcc_sensitivity'], 'icc_specificity': prim['icc_specificity']},
           'protocol': '00 log section 17 (approved 17 Oct 2026) with amendments 17.9; no tuning on MCT-LTDiag; argmax decisions'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest_out = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
                for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest_out).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

def fmt(r):
    return (f"AUC {r['auc']:.3f} ({r['auc_delong_low']:.3f}-{r['auc_delong_high']:.3f}) | "
            f"HCC sens {r['hcc_sensitivity']:.2f} ({int(r['HCC_pred_HCC'])}/{int(r['n_HCC'])}) | "
            f"ICC spec {r['icc_specificity']:.2f} ({int(r['ICC_pred_ICC'])}/{int(r['n_ICC'])}) | "
            f"HCC->[H,I,C] {int(r['HCC_pred_HCC'])},{int(r['HCC_pred_ICC'])},{int(r['HCC_pred_CHCC'])} "
            f"ICC->[H,I,C] {int(r['ICC_pred_HCC'])},{int(r['ICC_pred_ICC'])},{int(r['ICC_pred_CHCC'])}")

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E6_{RUN_ID} | internal check passed: {GATE_OK} | included {len(INCLUDED)} '
      f'(HCC {int(Y.sum())}, ICC {int((1 - Y).sum())}) | technical failures {len(TECH_FAIL)}')
print('Facts:', json.dumps(FACTS_SUMMARY, default=str))
print('Orientation:', {k: (v['applied_to'], v['unclear_set_to_majority']) for k, v in orientation_summary.items()},
      '| internal ref medians:', {k: round(REF[k], 2) for k in ['median_liver_i', 'median_bone_j', 'median_bulk_k']})
print(f"PRIMARY  {PRIMARY[0]}, primary preprocessing: AUC {prim['auc']:.3f} (DeLong {prim['auc_delong_low']:.3f}-"
      f"{prim['auc_delong_high']:.3f}; bootstrap {prim['auc_boot_low']:.3f}-{prim['auc_boot_high']:.3f})")
print('\n--- all models, primary preprocessing, all patients ---')
for model in MODEL_ORDER:
    r = RESULTS[(RESULTS['model'] == model) & (RESULTS['variant'] == 'primary') & (RESULTS['subset'] == 'all')].iloc[0]
    print(f'{model:42s} {fmt(r)}')
print('\n--- AUC by preprocessing variant (sensitivity analyses) and without prior treatment ---')
table = RESULTS[RESULTS['subset'] == 'all'].pivot(index='model', columns='variant', values='auc').reindex(MODEL_ORDER)[VARIANTS]
table['S3_no_prior_treatment'] = RESULTS[RESULTS['subset'] == 'no_prior_treatment'].set_index('model')['auc'].reindex(MODEL_ORDER)
print(table.round(3).to_string())
print('\n--- paired AUC differences, primary preprocessing (bootstrap 95% CI, DeLong p) ---')
print(PAIRED.round(3).to_string(index=False))
print('\nRadiomics median-filled fraction:', {k: round(v, 4) for k, v in IMPUTED.items()})
print('Versions:', VERSIONS)
print('==============================================================')